---
title: "Chapter -- Nearest Neighbors"
jupyter: python3

execute: 
  enabled: true
---

{{< chapter-actions >}}

## Introduction

Nearest Neighbors (k-NN) is one of the simplest and most intuitive supervised learning algorithms. The modern statistical formulation dates to the nearest-neighbor decision rule studied by Cover and Hart [@cover1967nearestneighbor]. Unlike many machine learning methods, k-NN does **not** estimate a parametric prediction equation during training. It retains the training observations and performs most of its work when a prediction is requested.

For this reason, k-NN is known as an **instance-based** or **lazy learning** algorithm.

Suppose the training set is

$$
\mathcal D= \left\{(\mathbf x_1,y_1), (\mathbf x_2,y_2), \ldots, (\mathbf x_n,y_n) \right\},
$$

where each observation consists of

- a predictor vector $\mathbf{x}_i$,
- a response $y_i$.

Given a new observation $\mathbf{x}$, the algorithm proceeds as follows:

1. Compute the distance from $\mathbf{x}$ to every observation in the training set.
2. Identify the $k$ closest observations.
3. Aggregate their responses to produce the prediction.

For **classification**, aggregation corresponds to a majority vote.

For **regression**, aggregation is usually the arithmetic mean of the neighboring responses.

::: {.callout-note}
## Local similarity assumption

Nearest Neighbors relies on a simple assumption:

> observations that are close in the feature space are likely to have similar responses.

Everything in k-NN follows from this idea.
:::

::: {.callout-note}
## Learning objectives

After completing this chapter, you should be able to:

- explain classification and regression with nearest neighbors;
- distinguish fixed-cardinality k-NN neighborhoods from fixed-radius neighborhoods;
- explain how scaling, distance, weighting, and neighborhood size affect predictions;
- build a leakage-safe `Pipeline` with `StandardScaler` and a nearest-neighbor estimator;
- jointly tune neighborhood size, weighting, and Minkowski distance with cross-validation;
- reserve the test set for one final evaluation;
- evaluate nearest-neighbor regression with MAE, RMSE, and $R^2$;
- diagnose ties, empty radius neighborhoods, high-dimensional data, and slow prediction.
:::

## k-Nearest Neighbor Classification

The simplest version of the algorithm uses

$$
k=1,
$$

meaning that only the closest observation is considered.

@fig-knn-k1 illustrates the prediction process.

In [ ]:
#| label: fig-knn-k1
#| fig-cap: Classification using the nearest neighbor.
#| code-fold: true
#| code-summary: Show code

import numpy as np
import matplotlib.pyplot as plt

class0 = np.array([
    [1.0,1.0],
    [1.3,2.2],
    [2.0,1.4],
    [2.3,2.5],
    [1.6,3.0]
])

class1 = np.array([
    [5.2,4.8],
    [4.8,5.7],
    [6.1,5.2],
    [5.7,6.0],
    [4.6,4.4]
])

new_point = np.array([3.2,3.3])

points = np.vstack([class0,class1])

distances = np.linalg.norm(
    points-new_point,
    axis=1
)

nearest = np.argmin(distances)

fig, ax = plt.subplots(figsize=(6,5))

ax.scatter(
    class0[:,0],
    class0[:,1],
    s=80,
    marker="o",
    label="Class 0"
)

ax.scatter(
    class1[:,0],
    class1[:,1],
    s=80,
    marker="^",
    label="Class 1"
)

ax.scatter(
    new_point[0],
    new_point[1],
    marker="*",
    s=250,
    color="red",
    label="New observation"
)

ax.plot(
    [new_point[0],points[nearest,0]],
    [new_point[1],points[nearest,1]],
    "--",
    linewidth=2
)

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
ax.grid(alpha=.3)

plt.show()

In @fig-knn-k1, the red star represents a new observation whose class is unknown. Shape as well as color distinguishes the two known classes, so the plot remains interpretable when colors are difficult to distinguish.

The dashed line connects the new observation with its closest training instance.

Since

$$
k=1,
$$

the prediction is simply the class label of this nearest observation.

If we increase the neighborhood size to

$$
k=5,
$$

the prediction is no longer determined by a single observation. Instead, the algorithm considers the five nearest neighbors and predicts the class receiving the majority of votes.

In [ ]:
#| label: fig-knn-k5
#| fig-cap: Classification using the five nearest neighbors.
#| code-fold: true
#| code-summary: Show code

k = 5

nearest = np.argsort(distances)[:k]

fig, ax = plt.subplots(figsize=(6,5))

ax.scatter(
    class0[:,0],
    class0[:,1],
    s=80,
    marker="o",
    label="Class 0"
)

ax.scatter(
    class1[:,0],
    class1[:,1],
    s=80,
    marker="^",
    label="Class 1"
)

ax.scatter(
    new_point[0],
    new_point[1],
    marker="*",
    s=250,
    color="red",
    label="New observation"
)

for idx in nearest:
    ax.plot(
        [new_point[0],points[idx,0]],
        [new_point[1],points[idx,1]],
        "--",
        linewidth=1.6
    )

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
ax.grid(alpha=.3)

plt.show()

Compared with @fig-knn-k1, @fig-knn-k5 makes the prediction depend on several nearby observations rather than on a single point.

As the value of $k$ increases, predictions generally become more stable because they are less influenced by individual observations. However, choosing a value that is too large may oversmooth the decision boundary and ignore important local patterns.

## Distance Metrics

The concept of *nearest* depends entirely on the distance metric.

In Scikit-Learn's nearest-neighbor estimators, the defaults `metric="minkowski"` and `p=2` make the effective distance Euclidean [@scikitLearn2025]:

$$
d(\mathbf{x},\mathbf{y}) = \sqrt{\sum_{j=1}^{p}(x_j-y_j)^2}.
$$

Another common choice is the **Manhattan distance**

$$
d(\mathbf{x},\mathbf{y})=\sum_{j=1}^{p}|x_j-y_j|.
$$

Both are special cases of the **Minkowski distance**

$$
d(\mathbf{x},\mathbf{y})=\left(\sum_{j=1}^{m}|x_j-y_j|^q\right)^{1/q},
$$

where

- $q=1$ gives the Manhattan distance;
- $q=2$ gives the Euclidean distance.

For Minkowski distance to be a metric, $q\geq 1$. The mathematical symbol $q$ avoids confusing the exponent with the number of features $m$; Scikit-Learn exposes this exponent through the API parameter `p`.

::: {.callout-tip}
## Standardize the predictors

Because k-NN relies entirely on distances, predictors measured on different scales can dominate the distance calculation.

In practice, k-NN is almost always preceded by feature standardization or normalization.
:::

## The `KNeighborsClassifier` Estimator

Scikit-Learn implements the k-nearest neighbors algorithm through the `KNeighborsClassifier` class [@scikitLearn2025]. The canonical estimator used in this chapter places scaling and classification in one pipeline.

Like most supervised learning estimators, its workflow consists of three steps:

1. Create the estimator.
2. Fit the estimator using the training data.
3. Predict the labels of new observations.

To illustrate its use, we first generate a simple synthetic binary classification dataset.

In [ ]:
#| label: generate-knn-data

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_knn, y_knn = make_moons(n_samples=300, noise=0.25, random_state=42)

X_knn_train, X_knn_test, y_knn_train, y_knn_test = train_test_split(X_knn, y_knn, test_size=0.25, random_state=42, stratify=y_knn)

Only the training observations are shown in @fig-knn-training-data. The test predictors and labels are not inspected before final evaluation.

In [ ]:
#| label: fig-knn-training-data
#| fig-cap: Nonlinear binary classification dataset used throughout the chapter.
#| code-fold: true
#| code-summary: Show code

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    X_knn_train[y_knn_train == 0, 0],
    X_knn_train[y_knn_train == 0, 1],
    s=45,
    marker="o",
    label="Class 0"
)

ax.scatter(
    X_knn_train[y_knn_train == 1, 0],
    X_knn_train[y_knn_train == 1, 1],
    s=45,
    marker="^",
    label="Class 1"
)

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

Next, we create a canonical scaled classifier using three neighbors. Keeping `StandardScaler` inside the pipeline ensures that it is fitted only on the data available to each training operation.

In [ ]:
#| label: create-knn-classifier

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])

The model is fitted using the training observations.

In [ ]:
#| label: fit-knn-classifier

_ = knn_pipeline.fit(X_knn_train, y_knn_train)

Unlike Decision Trees or Logistic Regression, fitting does not estimate coefficients or construct a parametric decision rule. It stores the transformed training observations and, depending on `algorithm`, may also construct a search index such as a KD tree or ball tree. This is still called *lazy learning* because predictions are determined from stored instances at query time, not because `fit()` necessarily performs no work.

For classification, the estimator's `score()` method returns prediction accuracy,

$$
\text{Accuracy}=\frac{\text{Correct predictions}}{\text{Total predictions}}.
$$

::: {.callout-note}
## What happens during prediction?

When a new observation is presented, the classifier:

1. finds nearby training observations using the fitted search strategy;
2. identifies the three nearest neighbors;
3. aggregates class votes, uniformly or by distance;
4. predicts the class with the majority of votes.

Prediction is generally more expensive than fitting, although fitting can include scaling and search-index construction.
:::

## Main Hyperparameters

The most important hyperparameters of `KNeighborsClassifier` are:

| Hyperparameter | Description |
|---------------|-------------|
| `n_neighbors` | Number of nearest neighbors used for prediction. |
| `weights` | Uniform or distance-weighted voting. |
| `metric` | Distance metric used to compare observations. |
| `p` | Exponent of the Minkowski distance. |
| `algorithm` | Neighbor search strategy (`auto`, `kd_tree`, `ball_tree`, or `brute`). |

Among these, the most influential is

```python
n_neighbors
```

A small value of $k$ produces a highly flexible classifier that adapts closely to the training data.

Larger values of $k$ generate smoother decision boundaries because the prediction depends on a larger neighborhood instead of only a few observations.

The effect of choosing different values of $k$ will be explored in the next section.

## Effect of the Number of Neighbors

The hyperparameter `n_neighbors` controls the complexity of the classifier.

A small value of $k$ allows the decision boundary to closely follow the training observations, producing a highly flexible model. Increasing the number of neighbors smooths the decision boundary because each prediction is influenced by a larger portion of the training data.

To visualize this effect, we train three classifiers using different neighborhood sizes.

In [ ]:
#| label: knn-models

from sklearn.base import clone

knn_1 = clone(knn_pipeline).set_params(knn__n_neighbors=1)
knn_3 = clone(knn_pipeline).set_params(knn__n_neighbors=3)
knn_9 = clone(knn_pipeline).set_params(knn__n_neighbors=9)

_ = knn_1.fit(X_knn_train, y_knn_train)
_ = knn_3.fit(X_knn_train, y_knn_train)
_ = knn_9.fit(X_knn_train, y_knn_train)

The following helper function plots the decision boundary of any fitted classifier.

In [ ]:
#| label: knn-decision-boundary-function

import numpy as np
import matplotlib.pyplot as plt

def plot_decision_boundary(model, X, y, ax, title):

    x_min, x_max = X[:,0].min() - 0.8, X[:,0].max() + 0.8
    y_min, y_max = X[:,1].min() - 0.8, X[:,1].max() + 0.8

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25)

    ax.contour(xx, yy, Z, linewidths=1)

    ax.scatter(
        X[y == 0, 0], X[y == 0, 1], s=55, marker="o", label="Class 0"
    )

    ax.scatter(
        X[y == 1, 0], X[y == 1, 1], s=55, marker="^", label="Class 1"
    )

    ax.set_title(title)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")

@fig-knn-k-comparison compares the decision boundaries for three different values of $k$.

In [ ]:
#| label: fig-knn-k-comparison
#| fig-cap: Decision boundaries for different values of k.
#| code-fold: true
#| code-summary: Show code

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10,3.8),
    sharex=True,
    sharey=True
)

plot_decision_boundary(
    knn_1,
    X_knn_train,
    y_knn_train,
    axes[0],
    "k = 1"
)

plot_decision_boundary(
    knn_3,
    X_knn_train,
    y_knn_train,
    axes[1],
    "k = 3"
)

plot_decision_boundary(
    knn_9,
    X_knn_train,
    y_knn_train,
    axes[2],
    "k = 9"
)

axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

Several important observations can be made from @fig-knn-k-comparison.

- With **$k=1$**, the classifier creates a very irregular decision boundary that closely follows individual observations. Such models usually have **low bias but high variance**, making them sensitive to noise.

- With **$k=3$**, the boundary becomes smoother while still preserving the overall structure of the data. This often provides a good balance between flexibility and generalization.

- With **$k=9$**, predictions are influenced by many surrounding observations. The boundary becomes much smoother, but some local patterns disappear. Excessively large values of $k$ may therefore lead to **underfitting**.

The choice of $k$ illustrates one of the most common examples of the **bias-variance tradeoff** in machine learning.

Small values of $k$ tend toward lower bias and higher variance; large values tend toward higher bias and lower variance.

::: {.callout-tip}
## Choosing an appropriate value of $k$

There is no universally optimal value of $k$.

The best choice depends on the dataset and is usually determined using a validation set or cross-validation, as will be shown in the next section.
:::

## Selecting a Nearest-Neighbor Classifier

Neighborhood size, voting weights, and distance should be selected together rather than in isolated experiments. The search below consistently considers $k=1,\ldots,40$, both weighting schemes, and the Manhattan and Euclidean cases of Minkowski distance. `StratifiedKFold` is named explicitly so that every fold preserves class proportions and the split is reproducible.

In [ ]:
#| label: knn-grid-search

from sklearn.model_selection import GridSearchCV, StratifiedKFold

classification_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

classification_grid = {
    "knn__n_neighbors": range(1, 41),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski"],
    "knn__p": [1, 2]
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=classification_grid,
    cv=classification_cv,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

_ = grid_search.fit(X_knn_train, y_knn_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best mean CV accuracy: {grid_search.best_score_:.3f}")

Because the scaler is inside `knn_pipeline`, every candidate fits scaling parameters from each CV training fold only. `refit=True` then refits the selected complete pipeline on all training observations. Printing selected parameters is more informative and compact than displaying the estimator object itself.

::: {.callout-tip}
## Why use a pipeline?

Scaling before cross-validation would let validation-fold values influence feature means and standard deviations. A pipeline prevents that leakage and carries exactly the fitted preprocessing needed for future observations.
:::

## Distance-Weighted Neighbors

So far, every neighbor has contributed equally to the prediction. This corresponds to the default setting

```python
weights="uniform"
```

where each of the $k$ nearest neighbors casts one vote, regardless of how close it is to the query point.

However, it is often reasonable to assume that **closer observations should have a greater influence** than more distant ones.

Scikit-Learn supports this idea through the `weights` hyperparameter.

```python
KNeighborsClassifier(n_neighbors=11, weights="distance")
```

When `weights="distance"` is used, each neighbor ordinarily contributes proportionally to the inverse of its distance from the query point.

Conceptually, the weight assigned to neighbor $i$ is

$$ w_i=\frac{1}{d(\mathbf{x},\mathbf{x}_i)},
$$

where

- $d(\mathbf{x},\mathbf{x}_i)$ is the distance between the new observation and neighbor $i$.

Consequently,

- nearby observations receive large weights;
- distant observations receive small weights.

For classification, the predicted class is obtained by summing the weights associated with each class,

$$
\hat y=\arg\max_c \sum_{i\in N_k}w_i I(y_i=c),
$$

where

- $N_k$ denotes the set of the $k$ nearest neighbors;
- $I(\cdot)$ is the indicator function.

The expression requires special handling at $d=0$. Scikit-Learn gives all nonzero-distance observations zero influence when one or more exact matches are present, then aggregates only the zero-distance matches. This avoids division by zero and allows duplicated observations to vote together.

Uniform voting can tie, especially with an even $k$; weighted scores can tie as well. Scikit-Learn stores class labels in `classes_` order and resolves equal winning scores in favor of the class with the smaller stored index. A prediction can also depend on training order when multiple observations at the $k$-th boundary have exactly the same distance but not all can be included. Odd $k$ reduces binary uniform-vote ties but does not solve multiclass or equal-distance ambiguity.

The following code compares the standard voting scheme with distance-weighted voting.

In [ ]:
#| label: weighted-knn-models

uniform_knn = clone(knn_pipeline).set_params(
    knn__n_neighbors=11,
    knn__weights="uniform"
)
distance_knn = clone(knn_pipeline).set_params(
    knn__n_neighbors=11,
    knn__weights="distance"
)

_ = uniform_knn.fit(X_knn_train, y_knn_train)
_ = distance_knn.fit(X_knn_train, y_knn_train)

The corresponding decision boundaries are shown in @fig-weighted-knn.

In [ ]:
#| label: fig-weighted-knn
#| fig-cap: Uniform voting versus distance-weighted voting.
#| code-fold: true
#| code-summary: Show code

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10,4),
    sharex=True,
    sharey=True
)

plot_decision_boundary(
    uniform_knn,
    X_knn_train,
    y_knn_train,
    axes[0],
    "Uniform weights"
)

plot_decision_boundary(
    distance_knn,
    X_knn_train,
    y_knn_train,
    axes[1],
    "Distance weights"
)

axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

The two classifiers in @fig-weighted-knn produce similar decision boundaries, with differences near the class boundaries.

With **uniform voting**, every neighbor contributes equally, regardless of its distance.

With **distance-weighted voting**, nearby observations exert greater influence, making the classifier more responsive to the local structure of the data.

::: {.callout-note}
## When should distance weighting be used?

Distance weighting is particularly useful when the selected value of $k$ is relatively large.

In such cases, some of the neighbors may be considerably farther away than others. Assigning larger weights to nearby observations helps preserve local information while still benefiting from a larger neighborhood.
:::

There is no universally superior weighting scheme. The joint grid search above selects it together with $k$ and distance rather than fixing the other choices first.


## Radius Neighbors Classification

In the k-nearest neighbors algorithm, every prediction is based on a fixed number of neighboring observations. Regardless of whether the query point is located in a dense or sparse region of the feature space, the classifier always considers exactly $k$ neighbors.

An alternative strategy is to define the neighborhood using a **fixed radius** instead of a fixed number of observations.

For a query point $\mathbf{x}$, all training observations whose distance satisfies

$$
d(\mathbf{x},\mathbf{x}_i)\le r
$$

are considered neighbors, where $r$ is a user-defined radius.

@fig-radius-neighborhood illustrates this fixed geometric radius.

In [ ]:
#| label: fig-radius-neighborhood
#| fig-cap: Radius Neighbors considers all observations located inside a predefined radius.
#| code-fold: true
#| code-summary: Show code

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

class0 = np.random.normal(
    loc=[1.8, 2.2],
    scale=0.35,
    size=(18,2)
)

class1 = np.random.normal(
    loc=[3.6, 2.9],
    scale=0.35,
    size=(18,2)
)

query = np.array([2.6,2.5])

radius = 1.0

fig, ax = plt.subplots(figsize=(8,8))

ax.scatter(
    class0[:,0],
    class0[:,1],
    s=55,
    marker="o",
    label="Class 0"
)

ax.scatter(
    class1[:,0],
    class1[:,1],
    s=55,
    marker="^",
    label="Class 1"
)

ax.scatter(
    query[0],
    query[1],
    s=170,
    marker="*",
    color="red",
    label="Query point",
    zorder=5
)

circle = plt.Circle(
    query,
    radius,
    fill=False,
    linestyle="--",
    linewidth=2,
    color="red"
)

ax.add_patch(circle)

ax.text(
    query[0]+0.15,
    query[1]+radius+0.05,
    r"$r$",
    fontsize=13
)

ax.set_xlim(0.7,4.8)
ax.set_ylim(1.0,4.4)

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")

ax.legend()

ax.grid(alpha=0.3)

plt.show()

Unlike k-nearest neighbors, the number of observations contributing to each prediction is **not fixed**. The geometric radius remains fixed in the scaled feature space; its cardinality varies with the empirical arrangement and sampling density of the training observations.

- Dense regions may contain many neighbors inside the radius.

- Sparse regions may contain only a few neighbors.

It is possible for a query to have no neighbors at all. With the default `outlier_label=None`, `predict()` raises an error for such a query. Setting `outlier_label` to a known label or `"most_frequent"` forces a fallback prediction, but can hide that the model has no local evidence. In applications where abstention is allowed, inspect `radius_neighbors()` and return an explicit "no prediction" status for empty neighborhoods instead of silently assigning a class.

The algorithm implemented in Scikit-Learn is `RadiusNeighborsClassifier`. Because a radius has meaning only relative to feature units, scaling belongs inside its pipeline too.

In [ ]:
#| label: radius-neighbors-model

from sklearn.neighbors import RadiusNeighborsClassifier

radius_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("radius", RadiusNeighborsClassifier(
        radius=0.80,
        weights="uniform",
        outlier_label="most_frequent"
    ))
])

_ = radius_pipeline.fit(X_knn_train, y_knn_train)

The resulting decision boundary is shown in @fig-radius-neighbors-boundary.

In [ ]:
#| label: fig-radius-neighbors-boundary
#| fig-cap: Decision boundary obtained using Radius Neighbors Classification.
#| code-fold: true
#| code-summary: Show code

fig, ax = plt.subplots(figsize=(6,5))

plot_decision_boundary(
    radius_pipeline, X_knn_train, y_knn_train, ax, "Radius Neighbors"
)

plt.show()

The main conceptual difference between the two nearest-neighbor algorithms is summarized below.

| k-Nearest Neighbors | Radius Neighbors |
|:-------------------|:-----------------|
| Fixed number of neighbors | Fixed search radius |
| Number of neighbors is always $k$ | Number of neighbors varies |
| Geometric reach varies to obtain $k$ observations | Geometric reach is always $r$ |

Choosing the radius involves the same bias-variance tradeoff observed for $k$.

- A **small radius** considers only very nearby observations. Predictions become highly local and may exhibit high variance.

- A **large radius** includes many observations. The decision boundary becomes smoother, but the classifier may underfit the data.

Consequently, the radius should also be regarded as a hyperparameter and selected using cross-validation.

In [ ]:
#| label: radius-grid-search
#| code-fold: true
#| code-summary: "Show code"

from sklearn.model_selection import GridSearchCV

radius_grid = {
    "radius__radius": np.linspace(0.3, 1.5, 9),
    "radius__weights": ["uniform", "distance"]
}

radius_search = GridSearchCV(
    radius_pipeline,
    radius_grid,
    cv=classification_cv,
    scoring="accuracy"
)

_ = radius_search.fit(X_knn_train, y_knn_train)

print("Best parameters:", radius_search.best_params_)
print("Mean CV accuracy:", f"{radius_search.best_score_:.3f}")

::: {.callout-note}
## When should Radius Neighbors be preferred?

Radius Neighbors can be useful when a fixed notion of geometric proximity is meaningful. It does not adapt its radius to density: it uses the chosen scaled radius everywhere, resulting in more sampled observations in dense regions, fewer in sparse regions, and potentially none outside the training support.
:::

The weighted and radius sections above were training-only demonstrations; they did not inspect the test set. After completing all classification model development, we evaluate the selected k-NN pipeline on the independent test set exactly once:

In [ ]:
#| label: final-knn-test-performance

best_pipeline = grid_search.best_estimator_
test_accuracy = best_pipeline.score(X_knn_test, y_knn_test)

print(f"Final test accuracy: {test_accuracy:.3f}")

The test result estimates performance after the complete model-selection procedure. Looking at it repeatedly and revising the grid would turn the test set into validation data and make that estimate optimistic.

## k-Nearest Neighbors Regression

The nearest neighbors approach is not limited to classification problems. It can also be applied to **regression**, where the objective is to predict a continuous response rather than assigning observations to discrete classes.

The fundamental idea remains unchanged. Given a new observation, the algorithm first identifies its nearest neighbors. Instead of selecting the majority class, however, it predicts the response as the average of the neighboring target values.

Suppose that the $k$ nearest neighbors of a query point $\mathbf{x}$ have response values

$$
y_1,\ y_2,\ \ldots,\ y_k.
$$

The prediction is computed as

$$
\hat{y}(\mathbf{x}) = \frac{1}{k} \sum_{i=1}^{k} y_i.
$$

When distance weighting is enabled, closer observations contribute more strongly to the prediction than distant ones.

Scikit-Learn implements this algorithm through the `KNeighborsRegressor` class.

In [ ]:
#| label: generate-knn-regression-data

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

np.random.seed(42)

X_reg = np.sort(5 * np.random.rand(150, 1), axis=0)

y_reg = (np.sin(X_reg).ravel() + 0.25 * np.random.randn(150))

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

The following training-only illustration fits a scaled k-nearest neighbors regressor using five neighbors. It does not inspect the regression test set.

In [ ]:
#| label: knn-regressor

regression_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

_ = regression_pipeline.fit(X_reg_train, y_reg_train)

The fitted regression function is shown in @fig-knn-regression.

In [ ]:
#| label: fig-knn-regression
#| fig-cap: Prediction obtained using k-nearest neighbors regression.
#| code-fold: true
#| code-summary: Show code

X_plot = np.linspace(
    0,
    5,
    500
).reshape(-1,1)

y_plot = regression_pipeline.predict(X_plot)

fig, ax = plt.subplots(figsize=(8,5))

ax.scatter(
    X_reg_train,
    y_reg_train,
    s=25,
    alpha=0.7,
    label="Training observations"
)

ax.plot(
    X_plot,
    y_plot,
    linewidth=2,
    label="k-NN prediction"
)

ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.legend()

plt.show()

Unlike linear regression, the k-nearest neighbors algorithm does not estimate a global mathematical function relating the predictors to the response.

Instead, predictions are obtained directly from nearby observations, allowing the model to naturally approximate highly nonlinear relationships.

As in classification, the neighborhood size controls the flexibility of the regression function.

- Small values of $k$ produce highly flexible predictions that closely follow the training observations.

- Larger values generate smoother regression curves because each prediction averages a greater number of neighboring responses.

@fig-knn-regression-k illustrates this effect using only training data.

In [ ]:
#| label: fig-knn-regression-k
#| fig-cap: Effect of the neighborhood size on k-nearest neighbors regression.
#| code-fold: true
#| code-summary: Show code

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10,4),
    sharey=True
)

for ax, k in zip(axes, [1, 5, 25]):

    model = clone(regression_pipeline).set_params(knn__n_neighbors=k)

    _ = model.fit(
        X_reg_train,
        y_reg_train
    )

    y_hat = model.predict(X_plot)

    ax.scatter(
        X_reg_train,
        y_reg_train,
        s=18,
        alpha=0.65
    )

    ax.plot(
        X_plot,
        y_hat,
        linewidth=2
    )

    ax.set_title(f"$k={k}$")
    ax.set_xlabel("$x$")

axes[0].set_ylabel("$y$")

plt.tight_layout()
plt.show()

The same bias-variance tradeoff observed in classification also appears in regression.

- Small values of $k$ produce low bias but high variance because the fitted curve follows individual observations very closely.

- Large values of $k$ reduce variance by averaging many neighboring responses, but excessive smoothing may lead to underfitting.

Consequently, the value of $k$ should also be regarded as a hyperparameter and selected with a validation holdout or cross-validation. Cross-validation uses limited data more efficiently; a sufficiently large validation holdout can be cheaper. Neither may use the final test set.

For regression, model selection must use a regression metric. The next search minimizes mean absolute error through the `neg_mean_absolute_error` scoring convention and uses a named, shuffled `KFold`. It jointly selects neighborhood size, weighting, and Minkowski `p` on the regression training set.

In [ ]:
#| label: knn-regression-grid-search

from sklearn.model_selection import KFold

regression_cv = KFold(n_splits=5, shuffle=True, random_state=42)
regression_grid = {
    "knn__n_neighbors": range(1, 31),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski"],
    "knn__p": [1, 2]
}

regression_search = GridSearchCV(
    regression_pipeline,
    regression_grid,
    cv=regression_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    refit=True
)

_ = regression_search.fit(X_reg_train, y_reg_train)

print("Best parameters:", regression_search.best_params_)
print(f"Best mean CV MAE: {-regression_search.best_score_:.3f}")

After all choices are fixed, the regression test set is evaluated once. MAE is the average absolute error in target units, RMSE penalizes large errors more strongly, and $R^2$ compares squared error with the test-set mean baseline. Lower MAE and RMSE are better; $R^2=1$ is perfect, $R^2=0$ matches that baseline, and $R^2$ can be negative.

In [ ]:
#| label: final-knn-regression-test-performance

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_regression_pipeline = regression_search.best_estimator_
y_reg_test_pred = best_regression_pipeline.predict(X_reg_test)

test_mae = mean_absolute_error(y_reg_test, y_reg_test_pred)
test_rmse = mean_squared_error(y_reg_test, y_reg_test_pred) ** 0.5
test_r2 = r2_score(y_reg_test, y_reg_test_pred)

print(f"Final test MAE:  {test_mae:.3f}")
print(f"Final test RMSE: {test_rmse:.3f}")
print(f"Final test R^2:  {test_r2:.3f}")

::: {.callout-note}
## Classification versus Regression

The nearest neighbors algorithm uses exactly the same search procedure for both classification and regression.

The only difference lies in the aggregation step.

- **Classification** predicts the most common class among the neighbors.

- **Regression** predicts the average (or distance-weighted average) of the neighboring response values.

Everything else—the distance metric, neighborhood search, and hyperparameter tuning—remains identical.
:::

## Computational Complexity and Practical Considerations

One of the main advantages of k-NN is its simplicity. It does not estimate a parametric prediction equation. Fitting always validates and stores the data, however, and tree-based search strategies also build an index. Its cost is therefore low for brute-force search but not literally zero.

| Phase | Computational cost |
|:------|:-------------------|
| Training | Usually low; index construction can add work |
| Prediction | Potentially high |

Much of the computational burden is shifted to prediction. Brute-force search computes distances to all training observations; tree methods may avoid many of those comparisons.

If the training dataset contains $n$ observations and $p$ predictor variables, a straightforward nearest-neighbor search requires approximately

$$
\mathcal{O}(np)
$$

operations for each prediction.

Thus brute-force prediction scales linearly with training-set size per query, before neighbor selection and aggregation costs. Actual runtime also depends on batching, hardware, data representation, metric, and implementation.

### Efficient Neighbor Search

Scikit-Learn implements several algorithms to accelerate neighbor searches.

```python
KNeighborsClassifier(algorithm="auto")
```

The available search strategies are summarized below.

| Algorithm | Description |
|:-----------|:------------|
| `"brute"` | Computes the distance from the query point to every training observation. |
| `"kd_tree"` | Organizes the observations using a k-dimensional tree to reduce the search space. |
| `"ball_tree"` | Uses nested hyperspheres to partition the feature space efficiently. |
| `"auto"` | Chooses among the implemented strategies using Scikit-Learn heuristics. |

The default option

```python
algorithm="auto"
```

is a reasonable starting point, but it does not benchmark every strategy or guarantee the fastest one. Sparse input can force brute search, and tree pruning often loses effectiveness as dimensionality grows. For latency-sensitive or large applications, benchmark supported algorithms on representative fit and query workloads; approximate-neighbor libraries may be needed beyond Scikit-Learn's exact searches [@scikitLearn2025].


### The Curse of Dimensionality

Although k-nearest neighbors performs remarkably well in low-dimensional problems, its performance often deteriorates as the number of predictor variables increases.

This phenomenon is known as the **curse of dimensionality**.

As dimensionality increases,

- observations become increasingly far from one another;
- distances between observations become more similar;
- identifying meaningful nearest neighbors becomes more difficult.

@fig-knn-dimensionality summarizes this effect conceptually.

```{mermaid}
%%| echo: false
%%| label: fig-knn-dimensionality
%%| fig-cap: "Conceptual effect of increasing dimensionality on nearest-neighbor information."

flowchart LR

A["Low-dimensional space"]

A --> B["Neighbors are clearly separated"]

C["High-dimensional space"]

C --> D["Distances become increasingly similar"]

B --> E["Reliable nearest neighbors"]

D --> F["Neighborhood becomes less informative"]
```

As summarized in @fig-knn-dimensionality, the notion of "nearest" can become less informative in high-dimensional spaces. This is a tendency rather than a universal threshold; sample size, feature relevance, sparsity, and metric all matter.

### Feature Scaling

Because nearest neighbors rely entirely on distance calculations, the scale of the predictor variables plays a crucial role.

Suppose one variable represents annual income measured in dollars, while another represents age measured in years.

Without preprocessing, differences in income will dominate the distance calculation, making age contribute very little to the prediction.

For this reason, feature scaling is generally recommended before applying k-nearest neighbors.

Common preprocessing techniques include

- Standardization (`StandardScaler`);
- Min-Max Scaling (`MinMaxScaler`);
- Robust Scaling (`RobustScaler`).

When cross-validation is used, scaling should be incorporated into a **Pipeline** to avoid data leakage.

### Advantages and Limitations

The main strengths and weaknesses of k-nearest neighbors are summarized below.

| Advantages | Limitations |
|:------------|:------------|
| Simple and intuitive | Slow predictions on large datasets |
| No training optimization required | Sensitive to irrelevant features |
| Naturally handles nonlinear decision boundaries | Sensitive to feature scaling |
| Works for both classification and regression | Performance deteriorates in high-dimensional spaces |
| Easy to interpret | Requires storing the complete training dataset |

::: {.callout-tip}
## When should k-nearest neighbors be used?

k-NN is particularly effective when

- the dataset has a moderate number of observations;
- the number of predictor variables is relatively small;
- similar observations are expected to have similar responses;
- interpretability and simplicity are more important than prediction speed.

For very large datasets or problems involving hundreds or thousands of features, tree-based ensemble methods or gradient boosting algorithms often provide better scalability and predictive performance.
:::

## Practical Workflow

1. Define the prediction target, deployment population, and an appropriate classification or regression metric.
2. Create the final test split immediately. Use stratification for classification when appropriate, and use grouped or time-aware splitting when observations are not independent and identically distributed.
3. Lock the test set. Do not inspect its values, plot it, preprocess it separately, or use its score to revise choices.
4. Put scaling and the nearest-neighbor estimator in one pipeline. Add any imputation or feature transformation to that same pipeline.
5. Use a named CV splitter on the training data to jointly tune neighborhood size, weighting, and distance. Ensure no candidate $k$ exceeds the smallest CV training fold.
6. Inspect CV means and variability, runtime, and application-relevant failure modes. For radius neighbors, also measure the empty-neighborhood rate.
7. Let `GridSearchCV(refit=True)` refit the selected pipeline on all training data.
8. Evaluate the locked test set once with prespecified metrics, then report the complete procedure and selected parameters.

## Common Failures

::: {.callout-warning}
## Scaling before the split or cross-validation

Fitting a scaler on all observations leaks validation or test information. Fit every data-dependent transformation inside the pipeline.
:::

::: {.callout-warning}
## Revisiting the test set

Comparing grids, plots, or preprocessing choices after seeing test performance makes the test set part of model selection. Create a new untouched holdout if this has already happened.
:::

::: {.callout-warning}
## Tuning one choice at a time

The best $k$ can change with scaling, weights, and metric. Tune interacting choices jointly within a computationally defensible grid.
:::

::: {.callout-warning}
## Treating distance as meaningful by default

Irrelevant features, mixed units, duplicated rows, and high dimensionality can make neighborhoods misleading even when code runs successfully.
:::

::: {.callout-warning}
## Hiding empty radius neighborhoods

`outlier_label="most_frequent"` guarantees an output but may conceal unsupported queries. Monitor empties and use explicit abstention when the application permits it.
:::

::: {.callout-warning}
## Reporting only accuracy or $R^2$

Accuracy can hide class-specific errors. $R^2$ does not express error in target units. Select metrics that match costs and report complementary diagnostics such as confusion measures or MAE and RMSE.
:::

## Chapter Summary

- k-NN predicts from a fixed number of nearby stored observations; radius neighbors use a fixed geometric radius and therefore a variable number of observations.
- Fitting stores data and may construct a search index, while prediction performs the local search and aggregation.
- The default Scikit-Learn configuration uses Minkowski distance with `p=2`, which is equivalent to Euclidean distance.
- Scaling must be fitted inside a pipeline because units directly determine neighborhoods.
- Neighborhood size, weights, and distance interact and should be selected jointly using only training-fold validation.
- Uniform and distance-weighted votes can tie; exact zero-distance matches receive special handling under inverse-distance weighting.
- Radius methods need an explicit policy for empty neighborhoods, including a fallback label or abstention.
- Regression requires a validation metric for selection and a one-time test evaluation with interpretable measures such as MAE, RMSE, and $R^2$.
- Exact neighbor search can be costly, and both statistical and computational performance often deteriorate as dimensionality grows.

## Exercises

1. Reproduce the classification grid search with `n_neighbors=range(1, 21)` and compare its selected model, CV score, and runtime with the chapter's `1..40` search. Do not reevaluate the test set.
2. Replace `accuracy` with balanced accuracy on an imbalanced synthetic dataset. Explain how the selected hyperparameters change.
3. Add `p=3` to the Minkowski grid. State why it remains a valid metric and compare CV performance and runtime.
4. Construct a binary example with an exact uniform-vote tie. Verify how `classes_` ordering affects the prediction.
5. Create duplicated observations at zero distance with conflicting labels. Compare uniform and distance weighting and explain Scikit-Learn's result.
6. For the radius pipeline, calculate the proportion of validation queries with no neighbors across the radius grid. Compare a fallback label with an explicit abstention policy.
7. Compare `algorithm="brute"`, `"kd_tree"`, `"ball_tree"`, and `"auto"` on low- and moderate-dimensional synthetic data. Measure fit and prediction time without assuming `"auto"` wins.
8. Repeat the regression search with RMSE rather than MAE as the selection criterion. Compare the selected parameters using training-fold CV only.
9. Build a regression holdout workflow instead of cross-validation. Report holdout MAE, RMSE, and $R^2$, and discuss the tradeoff in estimate stability and runtime.
10. Add irrelevant noise features to the moons data and examine how scaling, CV accuracy, and selected distance change as dimensionality increases.